# 27 · School inequality: data coverage, district fix, area results, Gini and Theil — Part D, step D6 (schools)

**Inputs:** the per-cell school results of notebooks 25 (Kerala, 2.98 M cells, Sametham levels) and 26 (India,
140 M cells, national levels).

**1. District fix.** The geoBoundaries state (ADM1) and district (ADM2) layers do not nest exactly, so 0.3 M border
cells (2.7 M people) carry a district of the neighbouring state. The largest case is the 7 mandals moved from Telangana to
Andhra Pradesh in 2014, which ADM2 still draws inside Bhadradri. A district's state is the state holding most of
its people (Yanam set to Puducherry by name). Mismatched cells get the nearest district of their own state, then
the nearest sub-district of that district. Saved as `silver/cell_units_fixed` for every service.

**2. Data coverage.** UDISE 2022 pupils (all managements) ÷ WorldPop children per state and level. A state-level
under half the national value, or with no UDISE data, is flagged *UDISE incomplete* (Andhra Pradesh classes 11–12,
whose junior colleges are not listed; DNH & DD, no data). A state below half the national value for *primary*
(where nearly every child is enrolled) has an incomplete school list and is flagged at every level (Delhi: about
40 % of the expected pupils at every level). Flagged state-levels stay in the tables but are left
out of national indices (also reported with them).

**3. Area results** per state, district, sub-district (India) and district, local body (Kerala), weighted by
children: % beyond the distance norm, mean distance to the nearest school, places per child (main and enrolment).
One Spark pass gives additive sums per sub-district; every other table rolls up from them exactly.

**4. Inequality** (`scripts/bigdata/inequality.py`, self-checked): weighted **Gini** (binned in Spark, ratio
1.0001 per bin, checked against an exact sort on Kerala) and **Theil T** split into between / within states (India)
and districts / local bodies (Kerala), for places per child and nearest distance.

**Checks:** no cell or person lost; every district has one state; the state table equals notebook 26's; Gini binned =
exact within 1e-4; Theil between + within = total.

**Outputs** (`data/lake/gold/`, copied to `FINAL COMPLETE DATASET/results/schools/`): `school_data_coverage.csv`,
`school_india_by_{state,district,subdistrict}.csv`, `school_india_inequality.csv`, `school_gini_check.csv`,
`school_kerala_by_{district,local_body}.csv`, `school_kerala_inequality.csv`.

## Project folder (laptop or Colab)

In [1]:
# Project folder. Laptop: open the notebook from "PROJECTS IPYNB" (or the project root).
# Google Colab: upload the project folder to Google Drive ("My Drive") and set its name below.
from pathlib import Path
import sys
PROJECT_FOLDER = "Distance-Capacity-and-Inequality-in-Access-to-Government-and-Aided-Schools"  # Colab only: folder name inside My Drive
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive") / PROJECT_FOLDER
else:
    PROJECT = Path.cwd().parent if Path.cwd().name in ("notebooks", "PROJECTS IPYNB") else Path.cwd()
assert (PROJECT / "scripts").is_dir(), f"project folder not found: {PROJECT}"
sys.path.insert(0, str(PROJECT / "scripts"))

In [2]:
# Colab only: install packages that Colab does not have by default (does nothing on the laptop)
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz", "pymupdf", "pyogrio", "rasterio"], check=True)

## Install the Part D packages

In [3]:
# Part D packages, pinned in requirements-bigdata.txt (PySpark 3.5.9, apache-sedona 1.9.1, pyarrow).
# Colab also needs Java 17, which Spark runs on (the laptop already has it: JAVA_HOME).
import os
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT / "requirements-bigdata.txt")], check=True)
if IN_COLAB:
    subprocess.run("apt-get install -y -qq openjdk-17-jdk-headless > /dev/null", shell=True, check=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

## Start Spark + Sedona

In [4]:
import shutil
import time

import geopandas as gpd
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from bigdata.spark_session import get_spark

sys.path.insert(0, str(PROJECT / "scripts" / "bigdata"))
import inequality as ineq
import population_cells as pc
import school_capacity as sc
from build_india_health_facilities import fold

SILVER, GOLD = PROJECT / "data" / "lake" / "silver", PROJECT / "data" / "lake" / "gold"
RESULTS = PROJECT / "FINAL COMPLETE DATASET" / "results" / "schools"
LEVELS = list(sc.INDIA_LEVELS)          # India run (notebook 26)
LEVELS_K = ["lp", "up", "hs", "hss"]    # Kerala run (notebook 25)
VARS = ["main", "enrol"]                # places per child: norm-based (main) and enrolment (sensitivity)
assert shutil.disk_usage(PROJECT).free / 1e9 > 3, "less than 3 GB free: stop"
spark = get_spark(PROJECT, cores=6, memory="6g")
LON, LAT = pc.lonlat_sql(PROJECT / "data" / "raw" / "worldpop")
cells_all = spark.read.parquet(str(SILVER / "population_cells")).filter("state <> '(outside boundaries)'")
print("cores:", spark.sparkContext.defaultParallelism, "| Spark UI:", spark.sparkContext.uiWebUrl)

cores: 6 | Spark UI: http://127.0.0.1:4040


## 1. District fix

In [5]:
# District fix. The state (ADM1) and district (ADM2) layers of geoBoundaries do not nest exactly, and each cell got
# its codes from each layer separately (notebook 17), so a cell near a state border can carry a district of the
# neighbouring state. A district's state = the state holding most of its people (cell centres); one exception:
# Yanam is a district of Puducherry, but both layers are off there (ADM2 polygon 57 km2, ADM1 part 17 km2, official
# about 30 km2), so it is set by name. Cells whose district belongs to another state (or have none) get the nearest
# district of their own state; then cells whose sub-district lies in another district get the nearest sub-district
# of their (fixed) district. Cells keep their state. Result: silver/cell_units_fixed (changed cells only), used by
# every service's D6.
def parent_of(df, child, parent):
    t = df.filter(f"{child} > 0").groupBy(child, parent).agg(F.sum("pop").alias("pop")).toPandas()
    t["share"] = t["pop"] / t.groupby(child)["pop"].transform("sum")
    return t.sort_values("pop").groupby(child).tail(1).set_index(child)


t0 = time.time()
units = {u: pc.load_units(SILVER, f"{u}_code").set_crs(4326, allow_override=True)  # WGS84; the files carry no CRS tag
         for u in ["district", "subdistrict"]}
dpar = parent_of(cells_all, "district_code", "state")
yanam = units["district"].loc[units["district"]["unit_name"] == "Yanam", "code"]
assert len(yanam) == 1
dpar.loc[yanam.iloc[0], "state"] = "Puducherry"
assert set(units["district"]["code"]) == set(dpar.index), "a district has no populated cell"
print("district -> state: lowest population shares (Yanam set to Puducherry by name):")
print(dpar.assign(name=units["district"].set_index("code")["unit_name"]).nsmallest(5, "share")[["name", "state", "share"]].round(3).to_string())
d_sdf = spark.createDataFrame(pd.DataFrame({"district_code": dpar.index.to_numpy("int32"), "d_state": dpar["state"].to_numpy()}))
valid = cells_all.join(F.broadcast(d_sdf), "district_code").filter("d_state = state")
spar = parent_of(valid, "subdistrict_code", "district_code")
s_sdf = spark.createDataFrame(pd.DataFrame({"subdistrict_code": spar.index.to_numpy("int32"), "s_district": spar["district_code"].to_numpy("int32")}))
bad = (cells_all.join(F.broadcast(d_sdf), "district_code", "left").join(F.broadcast(s_sdf), "subdistrict_code", "left")
       .filter("d_state IS NULL OR d_state <> state OR s_district IS NULL OR s_district <> district_code")
       .selectExpr("cell_id", "state", "district_code", "subdistrict_code", "pop", f"{LON} AS lon", f"{LAT} AS lat").toPandas())
pts = gpd.GeoDataFrame(bad, geometry=gpd.points_from_xy(bad["lon"], bad["lat"]), crs=4326).to_crs(7755)
poly = {u: g.to_crs(7755) for u, g in units.items()}


def nearest_code(p, cand):
    """Code of the nearest candidate polygon for each point (0 m inside it); ties keep the first."""
    j = gpd.sjoin_nearest(p[["geometry"]], cand[["code", "geometry"]], distance_col="dist_m")
    j = j[~j.index.duplicated()]
    return j["code"], j["dist_m"]


fix = bad.drop(columns="geometry", errors="ignore").copy()
fix["district_old"], fix["subdistrict_old"] = fix["district_code"], fix["subdistrict_code"]
d_ok = fix["district_code"].map(dpar["state"]).eq(fix["state"])
fix["reason"] = np.where(fix["district_code"].eq(0), "no district", np.where(d_ok, "sub-district only", "district of another state"))
fix["district_moved_km"] = 0.0
for st, idx in fix.index[~d_ok].groupby(fix.loc[~d_ok, "state"]).items():
    code, dist = nearest_code(pts.loc[idx], poly["district"][poly["district"]["code"].map(dpar["state"]).eq(st)])
    fix.loc[idx, "district_code"], fix.loc[idx, "district_moved_km"] = code.astype("int32"), dist / 1000
s_ok = fix["subdistrict_code"].map(spar["district_code"]).eq(fix["district_code"])
fix["subdistrict_moved_km"] = 0.0
no_sub = []
for dc, idx in fix.index[~s_ok].groupby(fix.loc[~s_ok, "district_code"]).items():
    cand = poly["subdistrict"][poly["subdistrict"]["code"].map(spar["district_code"]).eq(dc)]
    if cand.empty:
        fix.loc[idx, "subdistrict_code"] = 0  # the district has no sub-district polygon of its own
        no_sub.append(dc)
        continue
    code, dist = nearest_code(pts.loc[idx], cand)
    fix.loc[idx, "subdistrict_code"], fix.loc[idx, "subdistrict_moved_km"] = code.astype("int32"), dist / 1000
# checks: every changed cell now has a district of its own state, and a sub-district inside that district (or none)
assert fix["district_code"].map(dpar["state"]).eq(fix["state"]).all(), "a cell still has a district of another state"
assert (fix["subdistrict_code"].map(spar["district_code"]).eq(fix["district_code"]) | fix["subdistrict_code"].eq(0)).all()
assert fix["cell_id"].is_unique
fix = fix[["cell_id", "state", "district_old", "subdistrict_old", "district_code", "subdistrict_code", "reason",
           "district_moved_km", "subdistrict_moved_km", "pop"]]
fix[["district_code", "subdistrict_code"]] = fix[["district_code", "subdistrict_code"]].astype("int32")
print(f"\ncells changed: {len(fix):,} ({fix['pop'].sum() / 1e6:.2f} M people) in {time.time() - t0:.0f} s")
print(fix.groupby("reason").agg(cells=("cell_id", "size"), people=("pop", "sum")).round(0).to_string())
moved = fix.loc[fix["reason"] != "sub-district only", "district_moved_km"]
print("district moved (km): median %.2f, 95 %% %.2f, max %.2f" % (moved.median(), moved.quantile(.95), moved.max()))
print("districts without own sub-districts (cells left without one):", len(no_sub),
      f"{fix.loc[fix['district_code'].isin(no_sub) & fix['subdistrict_code'].eq(0), 'pop'].sum():,.0f} people")
print("largest moves by state and old district:")
top = fix[fix["reason"] == "district of another state"].groupby(["state", "district_old"])["pop"].sum().nlargest(8).reset_index()
top["old_district"] = top["district_old"].map(units["district"].set_index("code")["unit_name"])
top["its_state"] = top["district_old"].map(dpar["state"])
print(top.round(0).to_string(index=False))
fix.to_parquet(SILVER / "cell_units_fixed", index=False)
assert len(pd.read_parquet(SILVER / "cell_units_fixed")) == len(fix)
print("written silver/cell_units_fixed")

district -> state: lowest population shares (Yanam set to Puducherry by name):
                      name                                     state  share
district_code                                                              
615                  Yanam                                Puducherry  0.521
111               Longding                         Arunāchal Pradesh  0.793
170              Bhadradri                                 Telangāna  0.826
41                   Daman  Dādra and Nagar Haveli and Damān and Diu  0.877
332            Lower Siang                         Arunāchal Pradesh  0.914



cells changed: 3,001,759 (33.40 M people) in 177 s
                             cells      people
reason                                        
district of another state   330172   2716538.0
no district                   3384     25676.0
sub-district only          2668203  30657738.0
district moved (km): median 0.24, 95 % 34.00, max 54.43


districts without own sub-districts (cells left without one): 2 63,923 people
largest moves by state and old district:
         state  district_old      pop   old_district                                its_state
Andhra Pradesh           170 292690.0      Bhadradri                                Telangāna
         Bihār           527 123927.0 Uttar Dinajpur                              West Bengal
   Uttarākhand           301  71597.0         Rampur                            Uttar Pradesh
       Gujarāt            41  64973.0          Daman Dādra and Nagar Haveli and Damān and Diu
Madhya Pradesh           340  56222.0     Pratapgarh                                Rājasthān
    Tamil Nādu           167  49677.0     Puducherry                               Puducherry
         Bihār            62  48964.0      Darjiling                              West Bengal
   Mahārāshtra           420  47307.0         Valsad                                  Gujarāt


written silver/cell_units_fixed


## 2. Data coverage of UDISE 2022

In [6]:
# Data coverage: are the UDISE 2022 schools complete? Pupils listed (all managements) / children of the level's ages
# (WorldPop). Flag a state-level as "udise_incomplete" when it is below half the national value, or has no UDISE data.
share = pd.read_parquet(SILVER / "school_demand_share").query("scope == 'india'")
kids = cells_all.groupBy("state").agg(F.sum("pop").alias("pop"), F.count("*").alias("cells"),
                                     *[F.sum(F.coalesce(F.expr(sc.demand_sql(lv)), F.lit(0.0))).alias(lv) for lv in LEVELS]).toPandas()
cov = kids.melt(id_vars=["state", "pop", "cells"], value_vars=LEVELS, var_name="level", value_name="children").assign(key=lambda d: d["state"].map(fold))
cov = cov.merge(share.assign(key=share["state_name"].map(fold))[["key", "level", "pupils_all", "pupils_ga"]], on=["key", "level"], how="left")
cov["coverage"] = cov["pupils_all"] / cov["children"]
has = cov["pupils_all"].notna()
national = cov[has].groupby("level")["pupils_all"].sum() / cov[has].groupby("level")["children"].sum()
cov["national_coverage"] = cov["level"].map(national)
cov["udise_incomplete"] = ~has | (cov["coverage"] < 0.5 * cov["national_coverage"])
# Nearly every child of primary age is enrolled, so primary coverage tests whether a state's school LIST is complete:
# a state below half the national primary coverage has an incomplete list, and is flagged at every level (Delhi).
low_list = set(cov.loc[cov["level"].eq("primary") & cov["udise_incomplete"], "state"])
cov.loc[cov["state"].isin(low_list), "udise_incomplete"] = True
assert cov["state"].nunique() == len(kids) and has.groupby(cov["state"]).all().sum() >= len(kids) - 1, "a state lost its UDISE match"
print("national coverage:", national.round(3).to_dict(), "| incomplete school list (all levels flagged):", sorted(low_list))
assert cov.loc[cov["state"].eq("Delhi"), "udise_incomplete"].all() and cov["udise_incomplete"].sum() == 9, "unexpected flags"  # Delhi 4 + DNH & DD 4 + AP higher secondary
print("\nflagged as UDISE incomplete (left out of national rankings and indices):")
print(cov[cov["udise_incomplete"]][["state", "level", "pupils_all", "children", "coverage", "national_coverage"]].round(3).to_string(index=False))
print("\ncoverage per state (pupils listed / children):")
print(cov.pivot(index="state", columns="level", values="coverage")[LEVELS].round(2).to_string())
FLAG = {lv: set(cov.loc[cov["udise_incomplete"] & cov["level"].eq(lv), "state"]) for lv in LEVELS}
TOTAL_POP, TOTAL_CELLS = kids["pop"].sum(), int(kids["cells"].sum())

national coverage: {'higher_secondary': 0.465, 'primary': 0.933, 'secondary': 0.711, 'upper_primary': 0.816} | incomplete school list (all levels flagged): ['Delhi', 'Dādra and Nagar Haveli and Damān and Diu']

flagged as UDISE incomplete (left out of national rankings and indices):
                                   state            level  pupils_all    children  coverage  national_coverage
Dādra and Nagar Haveli and Damān and Diu          primary         NaN   91175.804       NaN              0.933
                                   Delhi          primary    685258.0 1576916.499     0.435              0.933
Dādra and Nagar Haveli and Damān and Diu    upper_primary         NaN   52887.908       NaN              0.816
                                   Delhi    upper_primary    452769.0 1003809.615     0.451              0.816
Dādra and Nagar Haveli and Damān and Diu        secondary         NaN   40803.580       NaN              0.711
                                   Delhi        se

## Functions

In [7]:
fixcodes = spark.read.parquet(str(SILVER / "cell_units_fixed")).select(
    "cell_id", F.col("district_code").alias("d_fix"), F.col("subdistrict_code").alias("s_fix"))


def fixed(df):
    """An access table with the corrected district (and sub-district) codes."""
    df = df.join(F.broadcast(fixcodes), "cell_id", "left").withColumn("district_code", F.coalesce("d_fix", "district_code"))
    if "subdistrict_code" in df.columns:
        df = df.withColumn("subdistrict_code", F.coalesce("s_fix", "subdistrict_code"))
    return df.drop("d_fix", "s_fix")


def xlnx(c):
    return F.when(c > 0, c * F.log(c)).otherwise(F.lit(0.0))


def unit_sums(df, keys, levels, variants):
    """One pass: additive sums per unit (children W, children beyond the norm, and X = sum w*x, L = sum w*x*ln x for
    places per child of each variant and for the nearest-school distance in km). Everything else rolls up from these."""
    aggs = [F.count("*").alias("cells"), F.sum("pop").alias("pop")]
    for lv in levels:
        d, near = F.col(f"dem_{lv}"), F.col(f"nearest_m_{lv}") / 1000
        aggs += [F.sum(d).alias(f"W_{lv}"), F.sum(F.when(F.col(f"schools_in_reach_{lv}").isNull(), d).otherwise(0)).alias(f"beyond_{lv}"),
                 F.sum(d * near).alias(f"X_near_{lv}"), F.sum(d * xlnx(near)).alias(f"L_near_{lv}"),
                 F.sum(F.when(near.isNull(), 1).otherwise(0)).alias(f"nulls_near_{lv}")]
        for v in variants:
            a = F.coalesce(F.col(f"a_{v}_{lv}"), F.lit(0.0))  # no school in reach = 0 places per child
            aggs += [F.sum(d * a).alias(f"X_{v}_{lv}"), F.sum(d * xlnx(a)).alias(f"L_{v}_{lv}")]
    s = df.groupBy(*keys).agg(*aggs).toPandas()
    assert not s[[c for c in s if c.startswith("nulls_near_")]].to_numpy().any(), "a cell has no nearest-school distance"
    return s


def area_table(s, keys, levels, variants):
    """Children-weighted results per unit (same definitions as notebooks 25 and 26)."""
    g = s.groupby(keys, as_index=False).sum(numeric_only=True) if keys else s.sum(numeric_only=True).to_frame().T
    out = g[keys + ["cells", "pop"]].copy()
    for lv in levels:
        W = g[f"W_{lv}"]
        out[f"children_{lv}"] = W
        out[f"beyond_norm_%_{lv}"] = 100 * g[f"beyond_{lv}"] / W
        out[f"mean_nearest_km_{lv}"] = g[f"X_near_{lv}"] / W
        for v in variants:
            out[f"a_{v}_{lv}"] = g[f"X_{v}_{lv}"] / W
    return out


def theil_rows(s, scope, group, lv, variants, keep=None):
    """Theil T of each variable for one level, split between / within `group`."""
    rows = []
    x = s if keep is None else s[keep]
    for var in [*variants, "near"]:
        t, b, w, _ = ineq.theil(x.rename(columns={f"W_{lv}": "W", f"X_{var}_{lv}": "X", f"L_{var}_{lv}": "L"}), group)
        rows.append({"scope": scope, "level": lv, "variable": var if var == "near" else f"a_{var}", "theil": t,
                     f"theil_between_{group}": b, f"theil_within_{group}": w, "between_share_%": 100 * b / t if t else np.nan})
    return rows


def gini_bins(df, group, lv, variants):
    """Per group, variable and value bin: sum of children W and of children x value X (one pass per level)."""
    xs = {f"a_{v}": f"coalesce(a_{v}_{lv}, 0D)" for v in variants} | {"near": f"nearest_m_{lv} / 1000"}
    st = ", ".join(f"'{k}', {e}" for k, e in xs.items())
    return (df.selectExpr(f"{group} AS grp", f"dem_{lv} AS w", f"stack({len(xs)}, {st}) AS (variable, x)").filter("w > 0")
              .selectExpr("grp", "variable", "w", "x", f"{ineq.bin_sql('x')} AS bin")
              .groupBy("variable", "grp", "bin").agg(F.sum("w").alias("W"), F.sum(F.col("w") * F.col("x")).alias("X")).toPandas())

## 3. India: results per state, district and sub-district

In [8]:
# India: one pass over the 140 M cells (fixed codes) -> sums per sub-district -> district, state and India tables
t0 = time.time()
acc = fixed(spark.read.parquet(str(GOLD / "school_access_india")))
s = unit_sums(acc, ["state", "district_code", "subdistrict_code"], LEVELS, VARS)
print(f"sums per sub-district: {len(s):,} rows in {time.time() - t0:.0f} s")
assert int(s["cells"].sum()) == TOTAL_CELLS and abs(s["pop"].sum() / TOTAL_POP - 1) < 1e-9, "cells or people lost"
assert s["district_code"].map(dpar["state"]).eq(s["state"]).all(), "a district still mixes states"
names = {u: pc.load_units(SILVER, f"{u}_code").set_index("code")["unit_name"] for u in ["district", "subdistrict"]}
by_state = area_table(s, ["state"], LEVELS, VARS).sort_values("state")
by_district = area_table(s, ["state", "district_code"], LEVELS, VARS)
by_sub = area_table(s, ["state", "district_code", "subdistrict_code"], LEVELS, VARS)
for t in (by_district, by_sub):
    t.insert(2, "district", t["district_code"].map(names["district"]))
by_sub.insert(4, "subdistrict", by_sub["subdistrict_code"].map(names["subdistrict"]).fillna("(none)"))
for t in (by_state, by_district, by_sub):
    for lv in LEVELS:
        t[f"udise_incomplete_{lv}"] = t["state"].isin(FLAG[lv])
india = area_table(s, [], LEVELS, VARS).iloc[0]
print("districts:", len(by_district), "| sub-district rows:", len(by_sub), "| states:", len(by_state))
# regression check: the fix moves cells between districts only, so the state table must equal notebook 26's
old = pd.read_csv(GOLD / "school_access_india_by_state.csv").set_index("state")
new = by_state.set_index("state")[old.columns]
assert np.allclose(new.loc[old.index].to_numpy(float), old.to_numpy(float), rtol=1e-9, equal_nan=True), "state results changed"
print("state table = notebook 26 (check passed)")
cols = ["state"] + [f"{m}_{lv}" for lv in LEVELS for m in ("beyond_norm_%", "a_main")]
print(by_state[cols].round(3).to_string(index=False))

sums per sub-district: 6,812 rows in 92 s
districts: 735 | sub-district rows: 6812 | states: 36


state table = notebook 26 (check passed)
                                   state  beyond_norm_%_primary  a_main_primary  beyond_norm_%_upper_primary  a_main_upper_primary  beyond_norm_%_secondary  a_main_secondary  beyond_norm_%_higher_secondary  a_main_higher_secondary
             Andaman and Nicobar Islands                 27.763           3.206                        7.186                 3.373                    5.693             2.677                           6.893                    1.607
                          Andhra Pradesh                 34.066           1.180                       13.331                 1.212                    5.872             0.914                          59.067                    0.028
                       Arunāchal Pradesh                 31.514           2.331                       14.475                 2.019                   19.129             1.206                          26.416                    0.586
                                   

## 4. India: Gini and Theil

In [9]:
# Inequality, India. Theil T: India split between / within states, each state split between / within districts.
# Gini: binned in Spark (one pass per level), exact on the sorted bins. Both with all states, and without the
# state-levels flagged as UDISE incomplete.
t0 = time.time()
th, gi = [], []
for lv in LEVELS:
    keep = ~s["state"].isin(FLAG[lv])
    th += theil_rows(s, "India", "state", lv, VARS)
    th += theil_rows(s, "India excl. UDISE-incomplete", "state", lv, VARS, keep)
    for st in sorted(s["state"].unique()):
        th += theil_rows(s, st, "district_code", lv, VARS, s["state"].eq(st))
    b = gini_bins(acc, "state", lv, VARS)
    for var, g in b.groupby("variable"):
        gi.append({"scope": "India", "level": lv, "variable": var, "gini": ineq.gini(g)})
        gi.append({"scope": "India excl. UDISE-incomplete", "level": lv, "variable": var, "gini": ineq.gini(g[~g["grp"].isin(FLAG[lv])])})
        gi += [{"scope": st, "level": lv, "variable": var, "gini": ineq.gini(x)} for st, x in g.groupby("grp")]
    print(f"{lv}: {len(b):,} bins, {time.time() - t0:.0f} s", flush=True)
ineq_india = pd.DataFrame(gi).merge(pd.DataFrame(th).rename(columns={"theil_between_district_code": "theil_between_district",
                                                                    "theil_within_district_code": "theil_within_district"}),
                                    on=["scope", "level", "variable"], how="outer")
print("groups without a Gini (no children or no places):", ineq_india["gini"].isna().sum())
assert ineq_india["gini"].dropna().between(0, 1).all()
part = ineq_india.dropna(subset=["theil_between_state"])
assert np.allclose(part["theil_between_state"] + part["theil_within_state"], part["theil"], atol=1e-9)
part = ineq_india.dropna(subset=["theil_between_district"])
assert np.allclose(part["theil_between_district"] + part["theil_within_district"], part["theil"], atol=1e-9)
print("Theil: between + within = total for every row (check passed)")
show = ineq_india[ineq_india["scope"].str.startswith("India")][["scope", "level", "variable", "gini", "theil", "between_share_%"]]
print(show.round(3).to_string(index=False))

primary: 6,758,989 bins, 212 s


upper_primary: 6,210,516 bins, 440 s


secondary: 5,988,941 bins, 665 s


higher_secondary: 6,132,525 bins, 881 s


groups without a Gini (no children or no places): 0
Theil: between + within = total for every row (check passed)
                       scope            level variable  gini  theil  between_share_%
                       India higher_secondary  a_enrol 0.481  0.412           18.398
                       India higher_secondary   a_main 0.547  0.530           42.668
                       India higher_secondary     near 0.519  0.506           35.606
                       India          primary  a_enrol 0.588  0.661            1.035
                       India          primary   a_main 0.603  0.700            9.114
                       India          primary     near 0.473  0.445           12.848
                       India        secondary  a_enrol 0.409  0.292            5.956
                       India        secondary   a_main 0.471  0.386           34.855
                       India        secondary     near 0.445  0.344            8.112
                       India    upper

## Check: binned Gini = exact Gini (Kerala)

In [10]:
# Check: binned Gini (Spark) = exact Gini on a full sort, on the 2.98 M Kerala cells of the India table
k = pd.read_parquet(GOLD / "school_access_india" / "state=Kerala")
rows = []
for lv in LEVELS:
    w = k[f"dem_{lv}"].to_numpy()
    for var, x in {"a_main": k[f"a_main_{lv}"].fillna(0), "a_enrol": k[f"a_enrol_{lv}"].fillna(0),
                   "near": k[f"nearest_m_{lv}"] / 1000}.items():
        exact = ineq.gini_exact(x.to_numpy()[w > 0], w[w > 0])
        binned = ineq_india.query("scope == 'Kerala' and level == @lv and variable == @var")["gini"].iloc[0]
        rows.append({"level": lv, "variable": var, "exact": exact, "binned": binned, "abs_diff": abs(exact - binned)})
gcheck = pd.DataFrame(rows)
print(gcheck.to_string(index=False))
assert gcheck["abs_diff"].max() < 1e-4, "binned Gini differs from the exact Gini"
del k

           level variable    exact   binned     abs_diff
         primary   a_main 0.610215 0.610215 3.069114e-10
         primary  a_enrol 0.664694 0.664694 2.418163e-10
         primary     near 0.348482 0.348482 7.065650e-10
   upper_primary   a_main 0.383512 0.383512 6.074908e-10
   upper_primary  a_enrol 0.429758 0.429758 5.122579e-10
   upper_primary     near 0.347983 0.347983 7.091218e-10
       secondary   a_main 0.322512 0.322512 7.508595e-10
       secondary  a_enrol 0.362873 0.362873 6.412498e-10
       secondary     near 0.336899 0.336899 7.442224e-10
higher_secondary   a_main 0.249947 0.249947 1.069724e-09
higher_secondary  a_enrol 0.271004 0.271004 9.407941e-10
higher_secondary     near 0.339999 0.339999 7.361480e-10


## Kerala case study

In [11]:
# Kerala case study (notebook 25 table, Sametham levels): district fix, tables per district and local body, Theil
# between / within districts and local bodies, Gini. Inequality over Kerala cells only (the Mahe cells, Puducherry,
# stay in the tables as their own row).
accK = fixed(spark.read.parquet(str(GOLD / "school_access_kerala")))
sK = unit_sums(accK, ["state", "district_code", "local_body_code"], LEVELS_K, VARS)
before = spark.read.parquet(str(GOLD / "school_access_kerala")).agg(F.count("*"), F.sum("pop")).first()
assert int(sK["cells"].sum()) == before[0] and abs(sK["pop"].sum() / before[1] - 1) < 1e-9
assert sK["district_code"].map(dpar["state"]).eq(sK["state"]).all()
lb_names = pc.load_units(SILVER, "local_body_code").set_index("code")["unit_name"]
k_district = area_table(sK, ["state", "district_code"], LEVELS_K, VARS)
k_district.insert(2, "district", k_district["district_code"].map(names["district"]))
# one row per local body: OSM local bodies and geoBoundaries districts do not nest either, so a local body's state and
# district = the ones holding most of its people (slivers across a district line stay inside their local body)
k_lb = area_table(sK, ["local_body_code"], LEVELS_K, VARS)
home = (sK.groupby(["local_body_code", "state", "district_code"], as_index=False)["pop"].sum()
          .sort_values("pop").groupby("local_body_code").tail(1).set_index("local_body_code"))
k_lb.insert(0, "state", k_lb["local_body_code"].map(home["state"]))
k_lb.insert(1, "district_code", k_lb["local_body_code"].map(home["district_code"]))
k_lb.insert(2, "district", k_lb["district_code"].map(names["district"]))
k_lb.insert(4, "local_body", k_lb["local_body_code"].map(lb_names).fillna("(none: no OSM local body)"))
assert k_lb["local_body_code"].is_unique and abs(k_lb["pop"].sum() / sK["pop"].sum() - 1) < 1e-9
split = sK[sK["local_body_code"] > 0].groupby("local_body_code")["district_code"].nunique()
print(f"local bodies: {len(k_lb)} rows (incl. 'none') | {int((split > 1).sum())} straddle a district line; their home district holds most people")
kerala = sK["state"].eq("Kerala")
moved = fix[fix["state"].eq("Kerala") & fix["reason"].ne("sub-district only")]
print(f"Kerala cells given a Kerala district: {len(moved):,} ({moved['pop'].sum():,.0f} people)")
rows = []
for lv in LEVELS_K:
    b = gini_bins(accK.filter("state = 'Kerala'"), "state", lv, VARS)
    gk = {var: ineq.gini(g) for var, g in b.groupby("variable")}
    dist = {r["variable"]: r for r in theil_rows(sK, "Kerala", "district_code", lv, VARS, kerala)}
    lbs = {r["variable"]: r for r in theil_rows(sK, "Kerala", "local_body_code", lv, VARS, kerala & sK["local_body_code"].gt(0))}
    for var in gk:
        d, l = dist[var], lbs[var]
        rows.append({"scope": "Kerala", "level": lv, "variable": var, "gini": gk[var], "theil": d["theil"],
                     "theil_between_district": d["theil_between_district_code"], "theil_within_district": d["theil_within_district_code"],
                     "between_district_%": d["between_share_%"],
                     "theil_cells_with_local_body": l["theil"],  # cells in no OSM local body (0.23 M people) left out
                     "theil_between_local_body": l["theil_between_local_body_code"],
                     "theil_within_local_body": l["theil_within_local_body_code"], "between_local_body_%": l["between_share_%"]})
ineq_kerala = pd.DataFrame(rows)
assert ineq_kerala["gini"].between(0, 1).all()
assert np.allclose(ineq_kerala["theil_between_district"] + ineq_kerala["theil_within_district"], ineq_kerala["theil"], atol=1e-9)
assert np.allclose(ineq_kerala["theil_between_local_body"] + ineq_kerala["theil_within_local_body"],
                   ineq_kerala["theil_cells_with_local_body"], atol=1e-9)
print(ineq_kerala[["level", "variable", "gini", "theil", "between_district_%", "between_local_body_%"]].round(3).to_string(index=False))
cols = ["district"] + [f"{m}_{lv}" for lv in LEVELS_K for m in ("beyond_norm_%", "a_main")]
print("\nper district:")
print(k_district[cols].round(3).to_string(index=False))

local bodies: 1028 rows (incl. 'none') | 244 straddle a district line; their home district holds most people


Kerala cells given a Kerala district: 3,018 (40,010 people)


level variable  gini  theil  between_district_%  between_local_body_%
   lp  a_enrol 0.665  0.834              21.977                33.788
   lp   a_main 0.612  0.689              15.356                28.679
   lp     near 0.332  0.185               5.057                22.583
   up  a_enrol 0.437  0.324              46.511                71.707
   up   a_main 0.386  0.250              35.661                66.177
   up     near 0.349  0.206               6.605                27.100
   hs  a_enrol 0.402  0.278              44.902                74.196
   hs   a_main 0.348  0.206              36.647                70.006
   hs     near 0.338  0.190               6.053                25.478
  hss  a_enrol 0.297  0.144              56.233                86.916
  hss   a_main 0.254  0.106              38.856                81.364
  hss     near 0.340  0.193               4.340                27.781

per district:
          district  beyond_norm_%_lp  a_main_lp  beyond_norm_%_up  a_main_u

## Save

In [12]:
RESULTS.mkdir(parents=True, exist_ok=True)
out = {"school_data_coverage.csv": cov.drop(columns="key"), "school_india_by_state.csv": by_state,
       "school_india_by_district.csv": by_district, "school_india_by_subdistrict.csv": by_sub,
       "school_india_inequality.csv": ineq_india, "school_gini_check.csv": gcheck,
       "school_kerala_by_district.csv": k_district, "school_kerala_by_local_body.csv": k_lb,
       "school_kerala_inequality.csv": ineq_kerala}
for name, t in out.items():
    t.to_csv(GOLD / name, index=False)
    shutil.copy2(GOLD / name, RESULTS / name)
print("written to data/lake/gold and FINAL COMPLETE DATASET/results/schools:", list(out))
spark.stop()

written to data/lake/gold and FINAL COMPLETE DATASET/results/schools: ['school_data_coverage.csv', 'school_india_by_state.csv', 'school_india_by_district.csv', 'school_india_by_subdistrict.csv', 'school_india_inequality.csv', 'school_gini_check.csv', 'school_kerala_by_district.csv', 'school_kerala_by_local_body.csv', 'school_kerala_inequality.csv']
